# Binary classification using the tutorial regression tools
**Workflow:** encode the two classes as -1/+1 -> construct polynomial features -> fit one weight vector -> calculate one score -> take its sign.

Edit `X`, `y`, `X_test`, `order`, and `lmd`, then **Run All**. This follows Tutorial 6 Q4 with the manual inverse formulas from your polynomial/ridge notebooks. All imports and numerical functions already appear in Tutorials 3-6 and your existing notebooks.

Use `order = 1` for a linear classifier, a higher order for a polynomial classifier, and positive `lmd` only when ridge regularization is requested. The notebook displays the weights, scores, predicted classes, and accuracy.


In [ ]:
import numpy as np
from numpy.linalg import inv
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score


## 1. Enter raw features and binary labels
Supply one sample per row in `X`, without manually adding a constant column. Each entry of `y` must be **-1 or +1**, according to the class mapping in the question. These labels are not inherently False/True unless the question defines them that way.

This sign-based workflow uses **one target column**, so one-hot encoding is not needed. If your labels are 0/1, convert them to -1/+1 before training, for example with `y = 2 * y - 1`.

`y_test = None` skips test accuracy when the true labels are unknown. `tie_class = None` leaves an exactly zero score unresolved (displayed as 0). Set `tie_class` to -1 or +1 only if that matches the question's tie rule.


In [ ]:
# EDIT INPUTS - Tutorial 6 Q4 example, with the bias column removed.
X = np.array([[-1], [0], [0.5], [0.3], [0.8]], dtype=float)
y = np.array([1, 1, -1, 1, -1])
X_test = np.array([[-0.1], [0.4]], dtype=float)
y_test = None  # Or np.array([1, -1]) if these are supplied true labels.

order = 1  # 1 = linear; 2, 3, ... = polynomial.
lmd = 0.0  # 0 = ordinary regression; positive = ridge, if requested.
tie_class = None  # Optional: -1 or +1, following the question's rule.


## 2. Polynomial features and dimensions
For p raw features and order n, including the offset:

$$
d = \binom{p+n}{n} = \frac{(p+n)!}{p!\,n!}.
$$

The design matrix P (`X_poly`) is **m by d**, the target column y is **m by 1**, and the weight vector w is **d by 1**:

$$
\underbrace{s}_{m\times1} = \underbrace{P}_{m\times d}\underbrace{w}_{d\times1}.
$$

`PolynomialFeatures` adds the intercept. The fitted `poly` object must also be used for test features. Keep targets as columns to prevent accidental broadcasting.


In [ ]:
X = np.array(X, dtype=float)
y = np.array(y, dtype=float)
assert len(X.shape) == 2, "Use one sample per row; one feature uses [[a], [b], ...]."
assert len(y.shape) == 1 or (len(y.shape) == 2 and y.shape[1] == 1), "Use one binary label per sample."
y = y.reshape(-1, 1)
assert X.shape[0] == y.shape[0], "X and y must have the same number of samples."
assert set(y.ravel()) <= {-1, 1}, "Encode training labels as -1/+1 first."
assert lmd >= 0, "Lambda must be zero or positive."
assert tie_class is None or tie_class in (-1, 1), "Tie class must be None, -1, or +1."

poly = PolynomialFeatures(degree=order)
X_poly = poly.fit_transform(X)
raw_names = [f"x{i + 1}" for i in range(X.shape[1])]
terms = poly.get_feature_names_out(raw_names)
print("Polynomial terms:", terms)
print("Polynomial matrix shape:", X_poly.shape)


## 3. Train the regression score
The ordinary model minimizes squared error to the -1/+1 target column. The ridge variant adds a penalty:

$$
\|Pw-y\|_2^2 + \lambda\|w\|_2^2.
$$

Use the same primal/dual formulas as regression:

$$
\boxed{w_{\mathrm{primal}}=(P^TP+\lambda I_d)^{-1}P^Ty},\qquad
\boxed{w_{\mathrm{dual}}=P^T(PP^T+\lambda I_m)^{-1}y}.
$$

With lambda zero, these are the left/right inverse formulas when their rank conditions hold. If neither ordinary inverse exists, Tutorial 4's pseudoinverse returns the minimum-norm least-squares weights. With positive lambda, the penalized matrices are invertible even when P is rank deficient. The code chooses the smaller matrix to invert. As in Tutorials 5-6, ridge penalizes all coefficients, including the intercept, using an unnormalized sum of squared errors.


In [ ]:
m, d = X_poly.shape
rank = np.linalg.matrix_rank(X_poly)

# MANUAL IF/ELSE: fit the one-column binary target.
if lmd == 0 and rank < m and rank < d:
    # Already used in Tut4; neither ordinary inverse exists here.
    w = np.linalg.pinv(X_poly) @ y
    method = "Pseudoinverse (minimum-norm least squares)"
elif m >= d:
    w = inv(X_poly.T @ X_poly + lmd * np.identity(d)) @ X_poly.T @ y
    method = "Primal ridge" if lmd > 0 else "Left inverse"
else:
    w = X_poly.T @ inv(X_poly @ X_poly.T + lmd * np.identity(m)) @ y
    method = "Dual ridge" if lmd > 0 else "Right inverse"

print("Method:", method)
print("Polynomial matrix shape/rank:", X_poly.shape, rank)
print("Weight shape:", w.shape)


## 4. Display the weights
Each term has one coefficient. The term `1` is the offset. For one raw feature at order 1:

$$
s(x) = w_0 + w_1x.
$$

At higher orders, the score also includes the displayed powers and interactions. Keep full-precision weights for prediction; round only the displayed results.


In [ ]:
for i in range(len(terms)):
    print(f"{terms[i]}: {w[i, 0]:.6f}")


## 5. Convert scores to classes and check training accuracy

$$
\hat c(x)=\operatorname{sign}(s(x))=
\begin{cases}
+1, & s(x)>0,\\
-1, & s(x)<0.
\end{cases}
$$

The decision boundary is **s(x) = 0**. `np.sign(0)` returns 0, which is unresolved until a tie rule is supplied. If `tie_class` is set, the code applies it to exactly zero scores. Unresolved zeros count as incorrect in the reported accuracy.

Accuracy is the fraction of predicted class labels matching the true labels. Very small scores near the boundary can be sensitive to floating-point rounding; do not round scores before taking their sign.


In [ ]:
train_scores = X_poly @ w
train_labels = np.sign(train_scores).ravel()
if 0 in train_labels:
    if tie_class is None:
        print("Exactly zero training scores are unresolved (label 0); set the specified tie rule if available.")
    else:
        train_labels[train_labels == 0] = tie_class

train_accuracy = accuracy_score(y.ravel(), train_labels)
print("Training scores:", train_scores.ravel())
print("Training predictions:", train_labels)
print("Training true labels:", y.ravel())
print("Training accuracy:", train_accuracy)


## 6. Predict test samples

$$
s_{\mathrm{test}}=\phi(X_{\mathrm{test}})w,\qquad
\hat c_{\mathrm{test}}=\operatorname{sign}(s_{\mathrm{test}}).
$$

Enter test samples in the same raw feature order as training. Reuse `poly.transform` to add the same polynomial terms. The score is a continuous number, **not a probability**. For the tutorial example, the test scores are approximately 0.4444 and -0.1111, giving classes +1 and -1.

Compare predicted **labels**, rather than raw scores, with `y_test` when measuring classification accuracy. Supply test labels using the same -1/+1 encoding as training.


In [ ]:
X_test = np.array(X_test, dtype=float)
X_test_poly = poly.transform(X_test)
scores = X_test_poly @ w
predicted_classes = np.sign(scores).ravel()
if 0 in predicted_classes:
    if tie_class is None:
        print("Exactly zero test scores are unresolved (label 0); set the specified tie rule if available.")
    else:
        predicted_classes[predicted_classes == 0] = tie_class

print("Test scores:", scores.ravel())
print("Predicted classes:", predicted_classes)
if y_test is not None:
    y_test = np.array(y_test)
    assert len(y_test.shape) == 1 or (len(y_test.shape) == 2 and y_test.shape[1] == 1), "Use one class label per test sample."
    assert set(y_test.ravel()) <= {-1, 1}, "Encode test labels as -1/+1 too."
    test_accuracy = accuracy_score(y_test.ravel(), predicted_classes)
    print("Test accuracy:", test_accuracy)
